# Gloss-Only Training Notebook

This notebook trains only the **gloss recognition model**.

It does **not** use iSign.
It does **not** train English sentence prediction.
It does **not** do gloss-free training.

Training target:

```text
pose sequence -> gloss sequence
```

Example:

```text
1.npy -> hello you how?
```

## Required Kaggle Input

Add one Kaggle dataset containing:

```text
gloss_training_colab.zip
```

Inside the zip, you need:

```text
gloss_structure/
data_gloss/
  gloss.csv
  labels.csv
  poses/
    1.npy
    2.npy
    ...
```

No `data_iSign` is needed for this notebook.

In [ ]:
from pathlib import Path
import zipfile
import shutil
import os

INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/gloss_project')

shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('Kaggle input folders:')
for p in sorted(INPUT_DIR.iterdir()):
    print(' -', p)

In [ ]:
matches = list(INPUT_DIR.rglob('gloss_training_colab.zip'))
if not matches:
    raise FileNotFoundError('Could not find gloss_training_colab.zip under /kaggle/input')

ZIP_PATH = matches[0]
print('Using zip:', ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(WORK_DIR)

print('Extracted zip.')

# Find the folder that contains gloss_structure/train_phase1_gloss.py
matches = list(WORK_DIR.rglob('train_phase1_gloss.py'))
if not matches:
    raise FileNotFoundError('Could not find gloss_structure/train_phase1_gloss.py after extraction')

PROJECT_DIR = matches[0].parents[1]
os.chdir(PROJECT_DIR)

print('Using PROJECT_DIR:', PROJECT_DIR)
print('Current dir:', Path.cwd())

## Verify Gloss Dataset

In [ ]:
import pandas as pd
import numpy as np

gloss_csv = Path('data_gloss/gloss.csv')
pose_dir = Path('data_gloss/poses')

print('gloss.csv exists:', gloss_csv.exists())
print('pose dir exists:', pose_dir.exists())

if not gloss_csv.exists():
    raise FileNotFoundError('Missing data_gloss/gloss.csv')
if not pose_dir.exists():
    raise FileNotFoundError('Missing data_gloss/poses')

df = pd.read_csv(gloss_csv)
pose_files = sorted(pose_dir.glob('*.npy'))

print('Rows:', len(df))
print('Pose files:', len(pose_files))
print(df.head())

for p in pose_files[:5]:
    arr = np.load(p)
    print(p.name, arr.shape, arr.dtype)

Expected:

```text
Rows: 78
Pose files: 78
1.npy (T, 225) float32
```

## Check GPU

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU found. In Kaggle, enable Accelerator -> GPU T4/P100.')

## Confirm Gloss Training Script

In [ ]:
!python -m gloss_structure.train_phase1_gloss --help

You should see options like:

```text
--train_all
--save_every
--blank_bias_init
--activity_loss_weight
--activity_target
```

## Train Gloss Model

This trains:

```text
pose + velocity -> gloss sequence
```

Because the gloss dataset is tiny, this uses `--train_all`. That means all 78 videos are used to train the gloss model.

In [ ]:
!python -m gloss_structure.train_phase1_gloss \
  --manifest data_gloss/gloss.csv \
  --pose_dir data_gloss/poses \
  --label_column gloss \
  --out_dir /kaggle/working/gloss_phase1 \
  --epochs 80 \
  --batch_size 4 \
  --lr 1e-4 \
  --dropout 0.3 \
  --hidden_size 128 \
  --num_layers 1 \
  --blank_bias_init -1.0 \
  --activity_loss_weight 0.02 \
  --activity_target 0.08 \
  --train_all \
  --save_every 5 \
  --monitor_metric token_er \
  --patience 40 \
  --device cuda

## Check Saved Files

In [ ]:
!ls -lh /kaggle/working/gloss_phase1

Expected files:

```text
best_gloss_model.pth
gloss_vocab.json
norm_stats.npz
history.json
epoch_005.pth
epoch_010.pth
...
```

## Plot Gloss Structural Signal

This plots:

```text
q_t = 1 - P(blank)
```

for one gloss video.

In [ ]:
!python -m gloss_structure.plot_structure \
  --checkpoint /kaggle/working/gloss_phase1/best_gloss_model.pth \
  --norm_stats /kaggle/working/gloss_phase1/norm_stats.npz \
  --pose_file data_gloss/poses/1.npy \
  --out_png /kaggle/working/gloss_phase1/gloss_qt_1.png \
  --device cuda

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/gloss_phase1/gloss_qt_1.png'))

## Inspect Training History

In [ ]:
import json

with open('/kaggle/working/gloss_phase1/history.json', 'r') as f:
    history = json.load(f)

best = min(history, key=lambda x: x['val_token_er'])
print('Best epoch by token error:')
print(best)

print('\nLast 5 epochs:')
for row in history[-5:]:
    print(row)

## Zip Gloss Outputs

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/gloss_phase1_outputs', 'zip', '/kaggle/working/gloss_phase1')
print('Created /kaggle/working/gloss_phase1_outputs.zip')